In [31]:
import os
import json
import pandas as pd
import traceback


In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI



In [33]:
from dotenv import load_dotenv
load_dotenv()


True

In [34]:
KEY = os.getenv("GEMINI_API_KEY")


In [35]:
# 1. Standard LLMs (Primary + Lighter Fallback)
primary_llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", google_api_key=KEY, temperature=0.7)
fallback_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", google_api_key=KEY, temperature=0.7)

# Retry up to 4 times with exponential backoff on 503 errors, then fall back to flash-lite
llm = primary_llm.with_retry(stop_after_attempt=4).with_fallbacks([fallback_llm])

# 2. JSON-enforced LLMs for the MCQ step (Primary + Lighter Fallback)
primary_json_llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=KEY,
    temperature=0.7,
    model_kwargs={"response_mime_type": "application/json"}
)
fallback_json_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=KEY,
    temperature=0.7,
    model_kwargs={"response_mime_type": "application/json"}
)

json_llm = primary_json_llm.with_retry(stop_after_attempt=4).with_fallbacks([fallback_json_llm])

In [36]:
json_llm

RunnableWithFallbacks(runnable=RunnableRetry(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-google-genai': '4.5.2'}}, profile={'name': 'Gemini 3.8 Flash', 'release_date': '2026-09-02', 'last_updated': '2026-09-02', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-3.8-flash', client=<google.genai.client.Client object at 0x00000290A5BF4050>, default_metadata=(), model_kwargs={}, response_mime_type='application/json'), kwargs={}, config={}, config_facto

In [37]:
import pypdf
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import get_usage_metadata_callback

In [38]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

TEMPLATE = """
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}
"""

quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

TEMPLATE2 = """
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students. \
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students, \
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

# Fixed: pointed to TEMPLATE2 instead of TEMPLATE
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [39]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Quiz generation chain (uses json_llm for valid JSON output)
quiz_chain = quiz_generation_prompt | json_llm | StrOutputParser()

# 2. Quiz evaluation chain (uses standard llm for text review)
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

# 3. Combine both steps sequentially (replaces SequentialChain)
generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [30]:
file_path = r"F:\mcqgenrator\data.txt"
with open(file_path, 'r') as file:
    TEXT = file.read()

NUMBER = 5
SUBJECT = "biology"
TONE = "simple"

# Use get_usage_metadata_callback() instead of get_openai_callback()
with get_usage_metadata_callback() as cb:
    response = generate_evaluate_chain.invoke(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject": SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
    )

# Sum tokens across both steps (quiz generation + review)
prompt_tokens = sum(m.get("input_tokens", 0) for m in cb.usage_metadata.values())
completion_tokens = sum(m.get("output_tokens", 0) for m in cb.usage_metadata.values())
total_tokens = sum(m.get("total_tokens", 0) for m in cb.usage_metadata.values())

print(f"Total Tokens: {total_tokens}")
print(f"Prompt Tokens: {prompt_tokens}")
print(f"Completion Tokens: {completion_tokens}")
print(f"Per-Model Breakdown: {cb.usage_metadata}")

f:\mcqgenrator\env\Lib\site-packages\langchain_google_genai\chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Total Tokens: 2217
Prompt Tokens: 1138
Completion Tokens: 1079
Per-Model Breakdown: {'gemini-3.5-flash-lite': {'output_tokens': 1079, 'total_tokens': 2217, 'input_tokens': 1138, 'input_token_details': {'cache_read': 0}}}


In [42]:
from datetime import datetime
quiz_str = response.get("quiz")
quiz_dict = json.loads(quiz_str)

quiz_table_data = []
for key, value in quiz_dict.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

quiz_df = pd.DataFrame(quiz_table_data)
quiz_df.to_csv("machinelearning.csv", index=False)

# Print review and timestamp
print("Review:\n", response.get("review"))
print("\nSaved at:", datetime.now().strftime('%m_%d_%Y_%H_%M_%S'))
quiz_df

Review:
 ### Complexity Analysis
The current quiz relies too heavily on basic memorization and lacks analytical depth suitable for high school or college biology students. The questions are overly simplistic, testing trivial definitions rather than biological reasoning, synthesis, or application of concepts. 

***

### Updated Quiz

```json
{
  "1": {
    "mcq": "Which fundamental characteristic most reliably distinguishes living organisms from non-living matter?",
    "options": {
      "a": "The presence of complex carbon-based molecules",
      "b": "The ability to maintain homeostasis and undergo energy transformation",
      "c": "Response to gravitational and magnetic fields",
      "d": "Capacity for physical movement and growth"
    },
    "correct": "b"
  },
  "2": {
    "mcq": "How does the core theme of evolution account for both the unity and diversity observed in biological systems?",
    "options": {
      "a": "By demonstrating that all species independently developed id

,MCQ,Choices,Correct
0,What is biology?,a: The study of the earth's weather | b: The s...,b
1,What major theme explains the unity and divers...,a: Evolution | b: Gravity | c: Photosynthesis ...,a
2,What are all organisms made up of?,a: Rocks | b: Cells | c: Metals | d: Plastic,b
3,Approximately when did life emerge on Earth?,a: More than 3.7 million years ago | b: More t...,b
4,Which of the following are examples of prokary...,a: Plants and animals | b: Fungi and protists ...,c
